## Подготовка данных

In [41]:
import xml.etree.ElementTree as ET
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, classification_report
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
import json

Напишем парсер, выделяющий из xml следующие признаки:
- **original**: исходное слово;
- **subpart_of_speech**: часть речи слова;
- **form**: форма слова;
- **genesys**: род и одушевлённость слова;
- **semantics1**, **semantics2**: значение семантики слова;
- **nucleus**: находится ли слово под фразовым ударением;
- **is_capital**: начинается ли слово с заглавной буквы;
- **word**: слово после нормализации;
- **length**: длина слова после нормализации;
- **fonetic_words_before**: количество фонетических слов в предложении перед словом;
- **fonetic_words_after**: количество фонетических слов в предложении после слова;
- **word_num**: номер слова в предложении;
- **has_pause**: есть ли пауза после слова;
- **sintagma_num**: номер синтагмы слова;
- **pause**: длина паузы после слова;
- **sentence_len**: количество слов в предложении, в котором находится данное слово;
- **prev_word**: предыдущее текущему слово в предложении (нормализованное);
- **next_word**: следующее за текущим слово в предложении (нормализованное);
- **PunktBeg**: знак препинания перед словом;
- **PunktEnd**: знак препинания после слова;
- **EmphBeg**: эмфаза перед словом;
- **EmphEnd**: эмфаза после слова.

In [2]:
class Parser():
    def parse_text(self, xml_text):
        self.sintagma_num = 0
        parsed_words = []
        for child in xml_text:
            if child.tag == "sentence":
                parsed_words.extend(self.parse_sentence(child))

        return parsed_words

    def parse_sentence(self, xml_sentence):
        sentence_words = []
        words_count = 0
        fonetic_words_count = 0
        word = {}
        for i in range(len(xml_sentence)):
            child = xml_sentence[i]
            if child.tag == "word":
                word_info = self.parse_word(child)
                word.update(word_info)
                
                word["fonetic_words_before"] = fonetic_words_count
                word["word_num"] = words_count + 1
                word["has_pause"] = False
                word["sintagma_num"] = self.sintagma_num
                words_count += 1
                fonetic_words_count += self.check_if_fonetic(child)
                
                sentence_words.append(word)
                word = {}
            elif child.tag == "content":
                content_info = self.parse_content(child)
                word.update(content_info)
            elif child.tag == "pause":
                self.sintagma_num += 1
                sentence_words[-1]["has_pause"] = True
                if "time" in child.attrib:
                    sentence_words[-1]["pause"] = child.attrib["time"]

        for i in range(len(sentence_words)):
            sentence_words[i]["fonetic_words_after"] = fonetic_words_count -  sentence_words[i]["fonetic_words_before"] - 1
            sentence_words[i]["sentence_len"] = words_count
            sentence_words[i]["prev_word"] = sentence_words[i - 1]["word"] if i > 0 else None
            sentence_words[i]["next_word"] = sentence_words[i + 1]["word"] if i < len(sentence_words) - 1 else None

        return sentence_words
                
    def parse_word(self, xml_word):
        word_info = {}
        if "original" in xml_word.attrib:
            self.original = xml_word.attrib["original"]
        
        word_info["original"] = self.original
        dictitem = xml_word.find("dictitem")
        if dictitem is not None:
            self.__add_item(word_info, dictitem, "subpart_of_speech")
            self.__add_item(word_info, dictitem, "form")
            self.__add_item(word_info, dictitem, "genesys")
            self.__add_item(word_info, dictitem, "semantics1")
            self.__add_item(word_info, dictitem, "semantics2")

        word_info["nucleus"] = ("nucleus" in xml_word.attrib)
        word_info["is_capital"] = False
        word = []
        for child in xml_word:
            if child.tag == "letter":
                if ("flag" in child.attrib) and (int(child.attrib["flag"]) & 16 > 0):
                    word_info["is_capital"] = True
                word.append(child.attrib["char"])

        word_info["word"] = "".join(word)
        word_info["length"] = len(word_info["word"])

        return word_info

    def check_if_fonetic(self, xml_word):
        for child in xml_word:
            if (child.tag == "letter") and ("flag" in child.attrib) and (int(child.attrib["flag"]) & 1 > 0):
                    return True

        return False

    def parse_content(self, xml_content):
        content_info = {}
        self.__add_item(content_info, xml_content, "PunktBeg")
        self.__add_item(content_info, xml_content, "PunktEnd")
        self.__add_item(content_info, xml_content, "EmphBeg")
        self.__add_item(content_info, xml_content, "EmphEnd")
        return content_info
        
    
    def __add_item(self, info, xml_node, key):
        if key in xml_node.attrib:
            info[key] = xml_node.attrib[key]

In [3]:
xml = ET.parse("Kuprin_Aleksandr__Olesya.Result.xml")
xml_root = xml.getroot()
parser = Parser()
df = pd.DataFrame(parser.parse_text(xml_root))
df

,original,subpart_of_speech,form,genesys,semantics2,nucleus,is_capital,word,length,fonetic_words_before,...,pause,fonetic_words_after,sentence_len,prev_word,next_word,PunktEnd,semantics1,PunktBeg,EmphBeg,EmphEnd
0,А.,1,0,0,700,False,True,а,1,0,...,1,2,3,None,и,NaN,NaN,NaN,NaN,NaN
1,И.,1,0,0,700,False,True,и,1,1,...,1,1,3,а,куприн,NaN,NaN,NaN,NaN,NaN
2,Куприн,1,1,1,NaN,True,True,куприн,6,2,...,359,0,3,и,None,1,20,NaN,NaN,NaN
3,Олеся,1,1,2,1,True,True,олеся,5,0,...,1657,0,1,None,None,1,10,NaN,NaN,NaN
4,I,1,0,0,700,True,True,ай,2,0,...,644,0,1,None,None,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22247,любви.,1,2,5,NaN,True,False,любви,5,18,...,800,0,27,великодушной,None,1,NaN,NaN,NaN,NaN
22248,< 1898 >,0,1,0,NaN,False,False,тысяча,6,0,...,NaN,3,4,None,восемьсот,NaN,NaN,NaN,5,NaN
22249,< 1898 >,0,1,0,NaN,False,False,восемьсот,9,1,...,NaN,2,4,тысяча,девяносто,NaN,NaN,NaN,NaN,NaN
22250,< 1898 >,0,1,0,NaN,False,False,девяносто,9,2,...,NaN,1,4,восемьсот,восемь,NaN,NaN,NaN,NaN,NaN


Предобработаем признаки, содержащие нормализованные слова, с помощью LabelEncoder:

In [4]:
def transform(encoder, df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(lambda x: x if x in encoder.classes_ else "<x>")
    df_part[column_name] = encoder.transform(df_part[column_name])

Но перед этим разделим выборку на тренировочную и тестовую (этот шаг необходим только для оценки качества моделей перед прохождением теста, во время теста его нужно заменить на парсинг тестового файла (закомментированный код)):

In [5]:
train_df, test_df = train_test_split(df, test_size = 0.2)

# train_df = df
# test_xml = ET.parse("test.xml")
# test_xml_root = test_xml.getroot()
# parser = Parser()
# test_df = pd.DataFrame(parser.parse_text(xml_root))

In [6]:
encoder = LabelEncoder()
encoder.fit(train_df["word"])
encoder.classes_ = np.append(encoder.classes_, "<x>")
transform(encoder, train_df, "word")
transform(encoder, test_df, "word")
transform(encoder, train_df, "prev_word")
transform(encoder, test_df, "prev_word")
transform(encoder, train_df, "next_word")
transform(encoder, test_df, "next_word")

Выделим категориальные признаки (для дальнейшего обучения CatBoost) и приведём их к строковому типу:

In [7]:
cat_features = ["form", "genesys", "semantics1", "semantics2", "is_capital", "word", "prev_word", "next_word", "PunktEnd", "PunktBeg", "EmphBeg", "EmphEnd"]
for feature in cat_features:
    train_df[feature] = train_df[feature].apply(lambda x: str(x))
    test_df[feature] = test_df[feature].apply(lambda x: str(x))

## Предсказание длины паузы

Для предсказания длины паузы будем обучать регрессор на данных, содержащих значение для паузы:

In [8]:
train_pause_df = train_df[train_df["has_pause"] == True]
len(train_pause_df)

4279

Опустим признаки, которые нужно предсказывать в рамках задач (длина паузы и наличие фразового ударения), а также такие признаки, как флаг, определяющий, есть ли пауза после слова, исходное слово и номер синтагмы:

In [9]:
def get_pause_X(df_part):
    return df_part.drop(["nucleus", "pause", "has_pause", "original", "sintagma_num"], axis = 1)

In [10]:
def get_pause_y(df_part):
    return df_part["pause"]

In [11]:
train_pause_X = get_pause_X(train_pause_df)
train_pause_y = get_pause_y(train_pause_df)

Обучим CatBoostRegressor со стандартными параметрами, при этом указав ему, какие признаки являются категориальными:

In [12]:
regressor = CatBoostRegressor(cat_features = cat_features)
regressor.fit(train_pause_X, train_pause_y)

Learning rate set to 0.051515
0:	learn: 219.7574796	total: 209ms	remaining: 3m 28s
1:	learn: 212.9405538	total: 219ms	remaining: 1m 49s
2:	learn: 206.6403912	total: 252ms	remaining: 1m 23s
3:	learn: 200.7602382	total: 274ms	remaining: 1m 8s
4:	learn: 195.3276482	total: 284ms	remaining: 56.5s
5:	learn: 190.1364604	total: 314ms	remaining: 52s
6:	learn: 185.4943754	total: 323ms	remaining: 45.9s
7:	learn: 181.0243586	total: 343ms	remaining: 42.5s
8:	learn: 176.9137335	total: 391ms	remaining: 43s
9:	learn: 173.1499526	total: 411ms	remaining: 40.7s
10:	learn: 169.7664322	total: 443ms	remaining: 39.8s
11:	learn: 166.5653037	total: 477ms	remaining: 39.3s
12:	learn: 163.6459824	total: 509ms	remaining: 38.7s
13:	learn: 160.9417866	total: 543ms	remaining: 38.3s
14:	learn: 158.5852623	total: 553ms	remaining: 36.3s
15:	learn: 156.4363816	total: 572ms	remaining: 35.2s
16:	learn: 154.4785468	total: 598ms	remaining: 34.6s
17:	learn: 152.8257394	total: 613ms	remaining: 33.4s
18:	learn: 150.9653130	tota

In [13]:
test_pause_df = test_df[test_df["has_pause"] == True]
test_pause_X = get_pause_X(test_pause_df)
test_pause_y = get_pause_y(test_pause_df)
test_y_pred = regressor.predict(test_pause_X)

В качестве метрики посчитаем RMSQ:

In [15]:
root_mean_squared_error(test_pause_y, test_y_pred)

136.83728850285766

Модель выдаёт достаточно хорошее значение метрики.

## Предсказание фразового ударения

Для предсказания фразового ударения будем обучать классификатор на всех данных.

Опустим признаки, которые нужно предсказывать в рамках задач (длина паузы и наличие фразового ударения), а также такие признаки, как флаг, определяющий, есть ли пауза после слова, исходное слово и номер синтагмы (попытка обработать данные после предсказания классификатора и оставить флаг фразового ударения в словах, имеющих наибольшее значение вероятности метки True в каждой отдельной синтагме, привело только к ухудшению качества модели):

In [16]:
def get_nucleus_X(df_part):
    return df_part.drop(["nucleus", "pause", "has_pause", "original", "sintagma_num"], axis = 1)

In [17]:
def get_nucleus_y(df_part):
    return df_part["nucleus"]

In [18]:
train_nucleus_X = get_nucleus_X(train_df)
train_nucleus_y = get_nucleus_y(train_df)

Обучим CatBoostRegressor со стандартными параметрами, при этом указав ему, какие признаки являются категориальными:

In [19]:
classifier = CatBoostClassifier(cat_features = cat_features)
classifier.fit(train_nucleus_X, train_nucleus_y)

Learning rate set to 0.035227
0:	learn: 0.6400183	total: 90.4ms	remaining: 1m 30s
1:	learn: 0.5950276	total: 140ms	remaining: 1m 9s
2:	learn: 0.5539465	total: 184ms	remaining: 1m 1s
3:	learn: 0.5200066	total: 235ms	remaining: 58.5s
4:	learn: 0.4874265	total: 260ms	remaining: 51.8s
5:	learn: 0.4579037	total: 311ms	remaining: 51.6s
6:	learn: 0.4321186	total: 362ms	remaining: 51.3s
7:	learn: 0.4095236	total: 412ms	remaining: 51.1s
8:	learn: 0.3874207	total: 474ms	remaining: 52.2s
9:	learn: 0.3679801	total: 533ms	remaining: 52.8s
10:	learn: 0.3473613	total: 584ms	remaining: 52.5s
11:	learn: 0.3307220	total: 632ms	remaining: 52.1s
12:	learn: 0.3154636	total: 678ms	remaining: 51.5s
13:	learn: 0.3030643	total: 717ms	remaining: 50.5s
14:	learn: 0.2924028	total: 769ms	remaining: 50.5s
15:	learn: 0.2807917	total: 818ms	remaining: 50.3s
16:	learn: 0.2692346	total: 870ms	remaining: 50.3s
17:	learn: 0.2600577	total: 927ms	remaining: 50.6s
18:	learn: 0.2519201	total: 983ms	remaining: 50.7s
19:	learn

In [20]:
test_nucleus_X = get_nucleus_X(test_df)
test_nucleus_y = get_nucleus_y(test_df)
test_y_pred = classifier.predict(test_nucleus_X)

Посмотрим на метрики классификации:

In [21]:
print(classification_report(test_nucleus_y, test_y_pred))

              precision    recall  f1-score   support

       False       0.96      0.99      0.98      3296
        True       0.97      0.90      0.93      1155

    accuracy                           0.97      4451
   macro avg       0.97      0.94      0.96      4451
weighted avg       0.97      0.97      0.97      4451



Модель выдаёт достаточно хорошие значения метрик.

## Формирование JSON

Обновим в DataFrame столбцы "pause" и "nucleus" в соотвествии с предсказаниями регрессора и классификатора и сохраним необходимые столбцы в json в заданном формате:

In [54]:
test_df["pause"] = regressor.predict(get_pause_X(test_df))
test_df["pause"] = test_df.apply(lambda x: -1 if x["has_pause"] else round(x["pause"]), axis = 1)
test_df["nucleus"] = classifier.predict(get_nucleus_X(test_df))

In [57]:
json_df = test_df[["original", "nucleus", "pause"]]
json_df.columns = ["content", "phrasal_stress", "pause_len"]
words = json.loads(json_df.to_json(orient="records", force_ascii=False, indent=4))
result_json = [{"words": words}]
with open("result.json", 'w') as result_file:
    json.dump(result_json, result_file, ensure_ascii=False, indent=4)